In [1]:
# --------------------------------------------------------------
# run_periodograms.py
# --------------------------------------------------------------
"""
Periodogramas Lomb‑Scargle (con FAP) y PDM para los sectores
10, 11, 37, 63 y 90 del TIC 465945338.

Este script está pensado para ser copiado dentro de una celda
de Jupyter (el notebook `TIC_465945338.ipynb`).  Sólo tiene que
ejecutarse y guardará, en la carpeta actual, los archivos:

    periodogramas_sXX.png      – gráfico LS + PDM
    folds_sXX.png              – curvas plegadas (1×Pₗₛ, 2×Pₗₛ, Pₚₙₘ, 2×Pₚₙₘ)
    periodogramas_sXX.ecsv     – tabla ECSV con frecuencias, potencias y θ

Si prefieres usar los archivos *.txt* que ya tienes, el código
detecta automáticamente el formato y los lee sin problemas.
"""

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.timeseries import LombScargle
from tqdm.auto import tqdm

# --------------------------------------------------------------
# Parámetros de búsqueda / cálculo
# --------------------------------------------------------------
TIC = "TIC 465945338"
SECTORES = [10, 11, 37, 63, 90]        # ¡ajústalos si necesitas más!

# Periodograma
PMIN_D = 0.05            # periodo mínimo (d) → fmax = 1/PMIN_D
SAMPLES_PER_PEAK = 10
FAP_LEVELS = [0.1, 0.01, 0.001]
N_BOOT = 200
PDM_NBINS = 10
N_PERM = 100
PERM_STEP = 5
SEED = 42
SHOW = False             # cambiar a True para ver las figuras
rng = np.random.default_rng(SEED)

# --------------------------------------------------------------
# Funciones auxiliares
# --------------------------------------------------------------
def cargar(sec):
    """
    Lee la tabla corregida del sector ``sec``.
    Busca primero un ECSV (más completo) y, si no lo encuentra,
    carga el txt tabulado que ya tienes.
    """
    base = f"tic465945338_s{sec:02d}_qlp_corrected"
    try:
        tab = Table.read(f"{base}.ecsv", format="ascii.ecsv")
        print(f"Sector {sec}: leído {base}.ecsv")
    except Exception:
        # fallback al txt (tab‑separated)
        tab = Table.read(f"{base}.txt", format="ascii.basic", delimiter="\t")
        print(f"Sector {sec}: leído {base}.txt")
    q = np.asarray(tab["quality"])
    t = np.asarray(tab["time_btjd"], dtype=float)
    y = np.asarray(tab["flux_corr"], dtype=float)
    dy = np.asarray(tab["flux_corr_err"], dtype=float)
    mask = (q == 0) & np.isfinite(t) & np.isfinite(y) & np.isfinite(dy)
    return t[mask], y[mask], dy[mask]


def pdm_theta(t, y, periodos, nbins=PDM_NBINS):
    """Estadístico θ de Stellingwerf (1978). Menor → mejor."""
    n = len(y)
    var = np.var(y, ddof=1)
    t0 = t.min()
    theta = np.empty(len(periodos))
    for i, P in enumerate(periodos):
        fase = ((t - t0) / P) % 1.0
        idx = np.minimum((fase * nbins).astype(int), nbins - 1)
        cnt = np.bincount(idx, minlength=nbins)
        s1 = np.bincount(idx, weights=y, minlength=nbins)
        s2 = np.bincount(idx, weights=y * y, minlength=nbins)
        ok = cnt > 1
        ss = np.sum(s2[ok] - s1[ok] ** 2 / cnt[ok])
        gl = n - ok.sum()
        theta[i] = (ss / gl) / var if gl > 0 else np.nan
    return theta


def pdm_fap(t, y, periodos, theta_obs_min):
    """FAP por permutaciones (barajar destruye la correlación temporal)."""
    sub = periodos[::PERM_STEP]
    mins = np.empty(N_PERM)
    for k in range(N_PERM):
        mins[k] = np.nanmin(pdm_theta(t, rng.permutation(y), sub))
    return (np.sum(mins <= theta_obs_min) + 1) / (N_PERM + 1)


def plegar(ax, t, y, P, titulo, nb=50):
    """Curva plegada y mediana por bin de fase."""
    fase = ((t - t.min()) / P) % 1.0
    ax.scatter(fase, y, s=2, c="k", alpha=0.3)
    ax.scatter(fase + 1, y, s=2, c="k", alpha=0.3)
    edges = np.linspace(0, 1, nb + 1)
    idx = np.clip(np.digitize(fase, edges) - 1, 0, nb - 1)
    med = np.array([np.median(y[idx == b]) if np.any(idx == b) else np.nan for b in range(nb)])
    cen = 0.5 * (edges[:-1] + edges[1:])
    ax.plot(np.r_[cen, cen + 1], np.r_[med, med], "r-", lw=1.5)
    ax.set_title(titulo, fontsize=9)
    ax.set_xlabel("fase")
    ax.set_ylabel("flujo norm.")


# --------------------------------------------------------------
# Bucle principal (un sector a la vez)
# --------------------------------------------------------------
resumen = []

for sec in SECTORES:
    try:
        t, y, dy = cargar(sec)
    except FileNotFoundError:
        print(f"Sector {sec}: archivo no encontrado → se omite.")
        continue

    if len(t) == 0:
        print(f"Sector {sec}: sin datos útiles → se omite.")
        continue

    T = t.max() - t.min()
    fmin, fmax = 2.0 / T, 1.0 / PMIN_D
    print(f"\n=== Sector {sec} | N={len(t)} | baseline={T:.2f} d ===")

    # ---------------- Lomb‑Scargle + FAP ----------------
    ls = LombScargle(t, y, dy)
    freq, power = ls.autopower(
        minimum_frequency=fmin,
        maximum_frequency=fmax,
        samples_per_peak=SAMPLES_PER_PEAK,
    )
    kw = dict(minimum_frequency=fmin, maximum_frequency=fmax)
    i_ls = np.argmax(power)
    f_ls, p_ls = freq[i_ls], power[i_ls]
    P_ls = 1.0 / f_ls

    fap_baluev = ls.false_alarm_probability(p_ls, method="baluev", **kw)
    niv_baluev = ls.false_alarm_level(FAP_LEVELS, method="baluev", **kw)

    fap_boot = ls.false_alarm_probability(
        p_ls,
        method="bootstrap",
        method_kwds=dict(n_bootstraps=N_BOOT, random_seed=SEED),
        **kw,
    )
    niv_boot = ls.false_alarm_level(
        FAP_LEVELS,
        method="bootstrap",
        method_kwds=dict(n_bootstraps=N_BOOT, random_seed=SEED),
        **kw,
    )
    print(
        f"LS : P={P_ls:.6f} d  power={p_ls:.4f}  "
        f"FAP(baluev)={fap_baluev:.3e}  FAP(bootstrap)={fap_boot:.3e}"
    )

    # ---------------- PDM ----------------
    periodos = 1.0 / freq
    theta = pdm_theta(t, y, periodos)
    i_pdm = np.nanargmin(theta)
    P_pdm, th_min = periodos[i_pdm], theta[i_pdm]
    fap_pdm = pdm_fap(t, y, periodos, th_min)
    print(f"PDM: P={P_pdm:.6f} d  theta_min={th_min:.4f}  FAP(perm)={fap_pdm:.3e}")

    # ---------------- Guardar periodogramas ----------------
    tabper = Table(
        {
            "freq_cpd": freq,
            "period_d": periodos,
            "ls_power": power,
            "pdm_theta": theta,
        }
    )
    tabper.meta.update(TARGET=TIC, SECTOR=sec)
    tabper.write(f"periodogramas_s{sec:02d}.ecsv", format="ascii.ecsv", overwrite=True)

    # ---------------- Figura LS + PDM ----------------
    fig = plt.figure(figsize=(14, 7))
    gs = fig.add_gridspec(2, 3)

    # LS panel
    ax1 = fig.add_subplot(gs[0, :2])
    ax1.plot(periodos, power, "k", lw=0.7)
    for lev, nb, nbo in zip(FAP_LEVELS, niv_baluev, niv_boot):
        ax1.axhline(nb, ls="--", lw=0.8, c="tab:red", label=f"Baluev FAP={lev}")
        ax1.axhline(nbo, ls=":", lw=0.8, c="tab:blue", label=f"Bootstrap FAP={lev}")
    ax1.axvline(P_ls, c="tab:green", alpha=0.4)
    ax1.set_xscale("log")
    ax1.set_xlabel("Periodo [d]")
    ax1.set_ylabel("Potencia LS")
    ax1.set_title(
        f"Lomb‑Scargle sector {sec}: P={P_ls:.5f} d, FAP(Baluev)={fap_baluev:.1e}"
    )
    ax1.legend(fontsize=7, ncol=2)

    # PDM panel
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.plot(periodos, theta, "k", lw=0.7)
    ax2.axvline(P_pdm, c="tab:green", alpha=0.5)
    ax2.set_xscale("log")
    ax2.set_xlabel("Periodo [d]")
    ax2.set_ylabel(r"$\theta$ (PDM)")
    ax2.set_title(f"PDM: P={P_pdm:.5f} d, FAP(perm)={fap_pdm:.1e}")

    # Curvas plegadas (1×Pₗₛ, 2×Pₗₛ, Pₚₙₘ)
    plegar(fig.add_subplot(gs[1, 0]), t, y, P_ls, f"Plegada a Pₗₛ = {P_ls:.5f} d")
    plegar(fig.add_subplot(gs[1, 1]), t, y, 2 * P_ls, f"Plegada a 2 Pₗₛ = {2*P_ls:.5f} d")
    plegar(fig.add_subplot(gs[1, 2]), t, y, P_pdm, f"Plegada a Pₚₙₘ = {P_pdm:.5f} d")

    fig.suptitle(f"{TIC} – sector {sec}")
    fig.tight_layout()
    fig.savefig(f"periodogramas_s{sec:02d}.png", dpi=150)
    if SHOW:
        plt.show()
    plt.close(fig)

    # ---------------- Figura de plegado ampliada ----------------
    cands = [
        (P_ls, "Pₗₛ"),
        (2 * P_ls, "2 Pₗₛ"),
        (P_pdm, "Pₚₙₘ"),
        (2 * P_pdm, "2 Pₚₙₘ"),
    ]
    figf, axf = plt.subplots(2, 2, figsize=(12, 8))
    for ax, (P, nom) in zip(axf.ravel(), cands):
        plegar(ax, t, y, P, f"Sector {sec} – {nom} = {P:.5f} d")
    figf.suptitle(
        f"{TIC} – sector {sec}: curvas plegadas (rojo = mediana por bin)"
    )
    figf.tight_layout()
    figf.savefig(f"folds_s{sec:02d}.png", dpi=150)
    if SHOW:
        plt.show()
    plt.close(figf)

    # ----------------------------------------------------------
    # Guardar datos de resumen para esta iteración
    # ----------------------------------------------------------
    resumen.append(
        (
            sec,
            len(t),
            P_ls,
            p_ls,
            fap_baluev,
            fap_boot,
            P_pdm,
            th_min,
            fap_pdm,
        )
    )

# --------------------------------------------------------------
# Tabla resumen (una fila por sector)
# --------------------------------------------------------------
if resumen:
    cols = list(zip(*resumen))
    res = Table(
        cols,
        names=(
            "sector",
            "N",
            "P_LS_d",
            "LS_power",
            "FAP_LS_baluev",
            "FAP_LS_bootstrap",
            "P_PDM_d",
            "PDM_theta_min",
            "FAP_PDM_perm",
        ),
    )
    res.write("resultados_periodogramas.ecsv", format="ascii.ecsv", overwrite=True)
    print("\n--- Resumen guardado en resultados_periodogramas.ecsv ---")
    res.pprint_all()


Sector 10: leído tic465945338_s10_qlp_corrected.txt
Sector 10: sin datos útiles → se omite.
Sector 11: leído tic465945338_s11_qlp_corrected.txt

=== Sector 11 | N=961 | baseline=25.33 d ===
LS : P=3.958268 d  power=0.2215  FAP(baluev)=5.275e-49  FAP(bootstrap)=0.000e+00
PDM: P=7.676640 d  theta_min=0.4216  FAP(perm)=9.901e-03
Sector 37: leído tic465945338_s37_qlp_corrected.txt

=== Sector 37 | N=2840 | baseline=23.63 d ===
LS : P=3.634619 d  power=0.1843  FAP(baluev)=2.688e-122  FAP(bootstrap)=0.000e+00
PDM: P=7.620976 d  theta_min=0.5291  FAP(perm)=9.901e-03
Sector 63: leído tic465945338_s63_qlp_corrected.txt

=== Sector 63 | N=10280 | baseline=24.71 d ===
LS : P=2.574081 d  power=0.2153  FAP(baluev)=0.000e+00  FAP(bootstrap)=0.000e+00
PDM: P=7.488237 d  theta_min=0.5659  FAP(perm)=9.901e-03
Sector 90: leído tic465945338_s90_qlp_corrected.txt

=== Sector 90 | N=10858 | baseline=27.43 d ===
LS : P=2.563434 d  power=0.2212  FAP(baluev)=0.000e+00  FAP(bootstrap)=0.000e+00
PDM: P=7.619095